# PHẦN V — BOOTSTRAP CEPH CLUSTER

## Bản chi tiết theo đúng style PHẦN IV

Phần này triển khai Ceph cluster 3 node trên 3 server vật lý.

Ceph sẽ dùng để làm backend storage cho OpenStack:

- Glance image backend
- Cinder volume backend
- Nova ephemeral / RBD backend
- Cinder backup backend

## Mapping server thực tế

| Server | Hostname Ceph | IP Management | IP Cluster/Ceph | Disk OSD thực tế |
|---|---|---:|---:|---|
| Server A | `server-a` | `10.0.0.21` | `10.0.3.21` hoặc `10.0.0.21` | `/dev/sda5` |
| Server B | `server-b` | `10.0.0.22` | `10.0.3.22` hoặc `10.0.0.22` | `/dev/sda5` |
| Server C | `server-c` | `10.0.0.23` | `10.0.3.23` hoặc `10.0.0.23` | `/dev/sda5` |

## Lưu ý theo thực tế

Từ ảnh `lsblk`, disk layout:

```text
sda
├─sda1  1G      /boot/efi
├─sda2  2G      /boot
├─sda3  150G    OS LVM /
├─sda4  150G    vg-lab thin-pool cho VM
└─sda5  814.2G  partition trống/candidate cho Ceph OSD
```

Vì vậy trong notebook này dùng:

```bash
/dev/sda5
```

làm OSD disk/partition.

## Cảnh báo quan trọng

- Tuyệt đối không wipe nhầm `/dev/sda3` hoặc `/dev/sda4`.
- `/dev/sda3` đang chứa OS.
- `/dev/sda4` đang chứa `vg-lab/thin-pool` cho VM.
- `/dev/sda5` mới là phần dành cho Ceph OSD theo ảnh.
- Sai disk = mất OS hoặc mất toàn bộ VM pool.

## Chọn network cho Ceph

Theo tài liệu ban đầu, Ceph bootstrap dùng:

```bash
--mon-ip 10.0.3.21
--cluster-network 10.0.3.0/24
```

Tức là dùng `br-cluster`.

Tuy nhiên nếu hiện tại `br-cluster` chưa ổn định, có thể bootstrap tạm bằng `10.0.0.x` management network.

Notebook này ghi cả 2 phương án, nhưng khuyến nghị:

- Nếu `br-cluster` ping ổn: dùng `10.0.3.x`.
- Nếu chưa chắc routing/VXLAN: dùng `10.0.0.x` để giảm lỗi bootstrap.


## BƯỚC 19 — Chuẩn bị OS trên 3 server cho Ceph

## Mục tiêu

Trên cả 3 server vật lý cần chuẩn bị:

- Ceph CLI cơ bản
- LVM tool
- Disk tool
- SMART monitoring
- Chrony để đồng bộ thời gian
- `/etc/hosts` để các node resolve được tên nhau
- SSH passwordless từ Server A sang Server B/C

## Vì sao NTP quan trọng?

Ceph rất nhạy với lệch thời gian. Nếu clock skew, cluster có thể báo lỗi:

```text
clock skew detected
mon.X clock skew
```

Trước khi bootstrap, nên đảm bảo offset thấp, lý tưởng `< 0.1s`.


## BƯỚC 19.1 — Cài package trên cả 3 server

Chạy block này trên:

- Server A
- Server B
- Server C


In [ ]:
# ============================================================
# BƯỚC 19.1 — Cài package Ceph/tool trên CẢ 3 SERVER
# ============================================================

# Update package index
sudo apt update

# Cài tool cần thiết cho Ceph
# ceph-common: cung cấp ceph CLI
# lvm2: cần cho OSD/LVM
# gdisk: dùng sgdisk để zap disk
# smartmontools: kiểm tra sức khỏe disk
# chrony: đồng bộ thời gian
sudo apt install -y \
  ceph-common \
  lvm2 \
  gdisk \
  smartmontools \
  chrony \
  curl \
  wget \
  openssh-client \
  openssh-server

# Enable chrony ngay
sudo systemctl enable --now chrony

# Verify package
which ceph || true
which sgdisk
which smartctl
systemctl status chrony --no-pager


## BƯỚC 19.2 — Cấu hình `/etc/hosts` trên cả 3 server

Mục tiêu là để Ceph dùng hostname ổn định:

- `server-a`
- `server-b`
- `server-c`

Chạy trên cả 3 server.

Nếu file `/etc/hosts` đã có dòng cũ bị sai, nên kiểm tra trước bằng:

```bash
cat /etc/hosts
```


In [ ]:
# ============================================================
# BƯỚC 19.2 — Cấu hình /etc/hosts trên CẢ 3 SERVER
# ============================================================

# Backup /etc/hosts trước khi sửa
sudo cp /etc/hosts /etc/hosts.bak.$(date +%F-%H%M%S)

# Thêm mapping hostname Ceph
# Nếu đã có dòng cũ, cần tránh duplicate sai IP.
sudo tee -a /etc/hosts << EOF
10.0.0.21  server-a
10.0.0.22  server-b
10.0.0.23  server-c
EOF

# Verify
cat /etc/hosts | egrep "server-a|server-b|server-c"

# Test resolve hostname
getent hosts server-a
getent hosts server-b
getent hosts server-c


## BƯỚC 19.3 — Kiểm tra kết nối giữa 3 server

Chạy trên Server A.

Mục tiêu:

- Server A ping được Server B/C qua management.
- Nếu dùng `br-cluster`, Server A cũng phải ping được cluster IP của B/C.


In [ ]:
# ============================================================
# BƯỚC 19.3 — SERVER A: kiểm tra kết nối management
# ============================================================

# Ping management IP
ping -c 3 10.0.0.22
ping -c 3 10.0.0.23

# Ping hostname
ping -c 3 server-b
ping -c 3 server-c

# Nếu dùng br-cluster cho Ceph:
# Kiểm tra cluster network 10.0.3.x
ping -c 3 10.0.3.22 || true
ping -c 3 10.0.3.23 || true


## BƯỚC 19.4 — SSH passwordless từ Server A sang B/C

Cephadm cần SSH từ node bootstrap sang các node còn lại.

Ở bước này tạo SSH key riêng:

```bash
~/.ssh/ceph_deploy
```

Sau đó copy sang Server B/C.

Chạy trên **Server A**.


In [ ]:
# ============================================================
# BƯỚC 19.4 — SERVER A: tạo SSH key deploy Ceph
# ============================================================

# Tạo SSH key riêng cho thao tác deploy Ceph
# Nếu file đã tồn tại, không nên overwrite nếu đang dùng.
ssh-keygen -t ed25519 -f ~/.ssh/ceph_deploy -C "ceph-deploy@lab" -N ""

# Copy public key sang Server B và Server C
ssh-copy-id -i ~/.ssh/ceph_deploy.pub ubuntu@10.0.0.22
ssh-copy-id -i ~/.ssh/ceph_deploy.pub ubuntu@10.0.0.23

# Test SSH không cần password
ssh -i ~/.ssh/ceph_deploy ubuntu@10.0.0.22 hostname
ssh -i ~/.ssh/ceph_deploy ubuntu@10.0.0.23 hostname


## BƯỚC 19.5 — Đồng bộ thời gian trên cả 3 server

Chạy trên cả 3 server.

Kết quả mong muốn:

- Chrony đang chạy.
- `System time` offset nhỏ.
- Không có lỗi clock skew lớn.


In [ ]:
# ============================================================
# BƯỚC 19.5 — Đồng bộ NTP bằng chrony trên CẢ 3 SERVER
# ============================================================

sudo systemctl enable --now chrony

# Kiểm tra nguồn time
chronyc sources -v

# Kiểm tra tracking
chronyc tracking

# Gợi ý:
# Offset nên nhỏ, tốt nhất < 0.1s trước khi bootstrap Ceph.
chronyc tracking | grep "System time" || true

# Kiểm tra thời gian hệ thống
timedatectl


## BƯỚC 20 — Xác định và chuẩn bị OSD disk

## Mục tiêu

Ceph OSD cần disk/partition riêng.

Theo ảnh thực tế bạn gửi, candidate OSD là:

```bash
/dev/sda5
```

Dung lượng khoảng `814.2G`.

## Cảnh báo cực kỳ quan trọng

Không chạy wipe trên:

- `/dev/sda`
- `/dev/sda1`
- `/dev/sda2`
- `/dev/sda3`
- `/dev/sda4`

Vì:

- `/dev/sda3` chứa OS LVM `/`
- `/dev/sda4` chứa `vg-lab/thin-pool`
- `/dev/sda5` mới là phần còn lại dành cho OSD

## Nguyên tắc

Trên cả 3 server, phải xác nhận `/dev/sda5` đúng là partition dành cho OSD rồi mới wipe.


## BƯỚC 20.1 — Kiểm tra disk layout trên cả 3 server

Chạy trên cả 3 server.


In [ ]:
# ============================================================
# BƯỚC 20.1 — Kiểm tra disk layout trên CẢ 3 SERVER
# ============================================================

# Xem tổng quan disk/partition
lsblk

# Xem chi tiết disk chính
lsblk /dev/sda

# Xem mountpoint
lsblk -o NAME,SIZE,TYPE,FSTYPE,MOUNTPOINTS /dev/sda

# Xem LVM
sudo pvs
sudo vgs
sudo lvs

# Kỳ vọng:
# - OS nằm ở /dev/sda3 hoặc LV ubuntu-vg/ubuntu-lv
# - VM thin pool nằm ở /dev/sda4 / vg-lab
# - OSD candidate là /dev/sda5


## BƯỚC 20.2 — Kiểm tra `/dev/sda5` trước khi wipe

Chạy trên cả 3 server.

Mục tiêu là xác nhận `/dev/sda5` không đang mount, không thuộc VG đang dùng, không chứa dữ liệu cần giữ.


In [ ]:
# ============================================================
# BƯỚC 20.2 — Kiểm tra /dev/sda5 trước khi wipe
# ============================================================

# Kiểm tra partition /dev/sda5
lsblk /dev/sda5

# Kiểm tra filesystem/signature hiện có
sudo wipefs -n /dev/sda5

# Kiểm tra /dev/sda5 có đang mount không
findmnt /dev/sda5 || true

# Kiểm tra /dev/sda5 có thuộc LVM PV không
sudo pvs | grep sda5 || true

# Kiểm tra SMART disk tổng thể
sudo smartctl -H /dev/sda || true

# Nếu thấy /dev/sda5 đang có mountpoint hoặc thuộc VG, DỪNG LẠI.


## BƯỚC 20.3 — Wipe OSD disk/partition trên cả 3 server

Chỉ chạy khi đã chắc chắn `/dev/sda5` là partition dành cho OSD.

Chạy trên:

- Server A
- Server B
- Server C


In [ ]:
# ============================================================
# BƯỚC 20.3 — Wipe /dev/sda5 để làm Ceph OSD
# CẢNH BÁO: kiểm tra kỹ trước khi chạy
# ============================================================

# Xóa filesystem signature
sudo wipefs -a /dev/sda5

# Zap GPT/MBR signature trên partition
# Với partition /dev/sda5, sgdisk --zap-all đôi khi phù hợp hơn với whole disk.
# Nếu lệnh dưới báo warning với partition thì có thể bỏ qua và dùng wipefs là chính.
sudo sgdisk --zap-all /dev/sda5 || true

# Đảm bảo kernel reload partition table
sudo partprobe /dev/sda || true

# Verify sau wipe
sudo wipefs -n /dev/sda5
lsblk -f /dev/sda5


## BƯỚC 21 — Cài cephadm và bootstrap cluster

## Mục tiêu

Bootstrap Ceph cluster từ Server A.

Server A sẽ là node đầu tiên chạy:

- MON đầu tiên
- MGR đầu tiên
- Dashboard đầu tiên

Sau đó mới add Server B/C vào cluster.

## Chỉ chạy bước này trên Server A

Không chạy bootstrap trên Server B/C.

## Chọn network bootstrap

### Phương án A — Theo thiết kế ban đầu dùng `br-cluster`

Dùng khi `10.0.3.21/22/23` ping nhau ổn:

```bash
--mon-ip 10.0.3.21
--cluster-network 10.0.3.0/24
```

### Phương án B — An toàn hơn nếu br-cluster chưa chắc

Dùng management network:

```bash
--mon-ip 10.0.0.21
--cluster-network 10.0.0.0/24
```

Trong lab hiện tại, nếu bạn đang chắc chắn `10.0.0.x` ổn hơn, dùng phương án B.


## BƯỚC 21.1 — Cài cephadm trên Server A

Chạy trên **Server A**.


In [ ]:
# ============================================================
# BƯỚC 21.1 — SERVER A: cài cephadm
# ============================================================

# Tải cephadm bản Reef
curl --silent --remote-name --location \
  https://github.com/ceph/ceph/raw/reef/src/cephadm/cephadm

# Cấp quyền execute
chmod +x cephadm

# Thêm repo Ceph Reef
sudo ./cephadm add-repo --release reef

# Cài cephadm vào hệ thống
sudo ./cephadm install

# Verify
which cephadm
cephadm version || sudo cephadm version


## BƯỚC 21.2 — Bootstrap Ceph bằng br-cluster `10.0.3.x` khuyến nghị theo thiết kế

Chỉ chạy block này nếu:

- Server A có IP `10.0.3.21`
- Server B có IP `10.0.3.22`
- Server C có IP `10.0.3.23`
- Server A ping được `10.0.3.22` và `10.0.3.23`


In [ ]:
# ============================================================
# BƯỚC 21.2 — SERVER A: bootstrap Ceph bằng br-cluster
# Chỉ chạy nếu 10.0.3.x hoạt động ổn
# ============================================================

# Kiểm tra IP br-cluster trên Server A
ip addr show br-cluster

# Kiểm tra kết nối br-cluster sang B/C
ping -c 3 10.0.3.22
ping -c 3 10.0.3.23

# Bootstrap Ceph cluster
sudo cephadm bootstrap \
  --mon-ip 10.0.3.21 \
  --cluster-network 10.0.3.0/24 \
  --initial-dashboard-user admin \
  --initial-dashboard-password 'Admin@Ceph2025!'

# Quá trình thường mất 3–5 phút
# Dashboard URL dự kiến:
# https://10.0.3.21:8443


## BƯỚC 21.2B — Bootstrap Ceph bằng management network `10.0.0.x`

Dùng block này nếu bạn muốn bootstrap đơn giản hơn bằng management network.

Chỉ chọn **một trong hai**:

- BƯỚC 21.2 dùng `10.0.3.x`
- hoặc BƯỚC 21.2B dùng `10.0.0.x`

Không chạy cả hai.


In [ ]:
# ============================================================
# BƯỚC 21.2B — SERVER A: bootstrap Ceph bằng management network
# Dùng nếu br-cluster chưa ổn định
# ============================================================

# Kiểm tra IP management trên Server A
ip addr show br-mgmt

# Kiểm tra kết nối management sang B/C
ping -c 3 10.0.0.22
ping -c 3 10.0.0.23

# Bootstrap Ceph cluster
sudo cephadm bootstrap \
  --mon-ip 10.0.0.21 \
  --cluster-network 10.0.0.0/24 \
  --initial-dashboard-user admin \
  --initial-dashboard-password 'Admin@Ceph2025!'

# Dashboard URL dự kiến:
# https://10.0.0.21:8443


## BƯỚC 21.3 — Verify bootstrap trên Server A

Sau bootstrap, cluster ban đầu thường chỉ có 1 MON nên có thể `HEALTH_WARN`.

Điều này bình thường ở giai đoạn đầu.

Chạy trên Server A.


In [ ]:
# ============================================================
# BƯỚC 21.3 — SERVER A: verify bootstrap
# ============================================================

# Kiểm tra trạng thái cluster
sudo ceph -s

# Kiểm tra MON
sudo ceph mon stat

# Kiểm tra MGR
sudo ceph mgr stat

# Kiểm tra service đang chạy bởi cephadm
sudo ceph orch ps

# Kiểm tra dashboard service
sudo ceph mgr services

# Expected:
# - Có 1 MON ban đầu
# - Có 1 MGR ban đầu
# - health có thể là HEALTH_WARN vì chưa đủ MON/OSD


## BƯỚC 22 — Thêm Server B và Server C vào Ceph cluster

## Mục tiêu

Từ Server A, thêm Server B/C vào Ceph orchestrator.

Cephadm dùng SSH root để quản lý node.

Cần copy public key Ceph:

```bash
/etc/ceph/ceph.pub
```

sang root của Server B/C.

## Lưu ý quan trọng

- Lệnh `ssh-copy-id root@...` yêu cầu root SSH hoặc có cách nhập password root.
- Nếu root SSH bị disable, cần cấu hình tạm hoặc dùng `ceph cephadm get-pub-key` và copy thủ công.
- Cephadm host IP nên nhất quán với network đã bootstrap:
  - Nếu bootstrap bằng `10.0.3.x`, add host bằng `10.0.3.22/23`.
  - Nếu bootstrap bằng `10.0.0.x`, add host bằng `10.0.0.22/23`.


## BƯỚC 22.1 — Copy SSH key cephadm sang Server B/C

Chạy trên Server A.

### Nếu dùng br-cluster `10.0.3.x`


In [ ]:
# ============================================================
# BƯỚC 22.1 — SERVER A: copy key cephadm qua br-cluster
# Dùng nếu bootstrap bằng 10.0.3.x
# ============================================================

# Kiểm tra public key cephadm
sudo cat /etc/ceph/ceph.pub

# Copy key sang Server B/C qua cluster IP
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.3.22
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.3.23

# Test SSH root bằng key cephadm
sudo ssh -i /etc/ceph/cephadm-key root@10.0.3.22 hostname || true
sudo ssh -i /etc/ceph/cephadm-key root@10.0.3.23 hostname || true


### Nếu dùng management network `10.0.0.x`

Chỉ chạy block này nếu bootstrap bằng `10.0.0.21`.


In [ ]:
# ============================================================
# BƯỚC 22.1B — SERVER A: copy key cephadm qua management network
# Dùng nếu bootstrap bằng 10.0.0.x
# ============================================================

sudo cat /etc/ceph/ceph.pub

sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.0.22
sudo ssh-copy-id -f -i /etc/ceph/ceph.pub root@10.0.0.23

sudo ssh -i /etc/ceph/cephadm-key root@10.0.0.22 hostname || true
sudo ssh -i /etc/ceph/cephadm-key root@10.0.0.23 hostname || true


## BƯỚC 22.2 — Add Server B/C vào Ceph orchestrator

Chọn đúng block theo network bootstrap.

### Nếu dùng `10.0.3.x`


In [ ]:
# ============================================================
# BƯỚC 22.2 — SERVER A: add host bằng cluster IP
# Dùng nếu bootstrap bằng 10.0.3.x
# ============================================================

sudo ceph orch host add server-b 10.0.3.22
sudo ceph orch host add server-c 10.0.3.23

# Kiểm tra host list
sudo ceph orch host ls


### Nếu dùng `10.0.0.x`


In [ ]:
# ============================================================
# BƯỚC 22.2B — SERVER A: add host bằng management IP
# Dùng nếu bootstrap bằng 10.0.0.x
# ============================================================

sudo ceph orch host add server-b 10.0.0.22
sudo ceph orch host add server-c 10.0.0.23

# Kiểm tra host list
sudo ceph orch host ls


## BƯỚC 22.3 — Deploy MON trên cả 3 server

Mục tiêu là có 3 MON:

```text
server-a
server-b
server-c
```

Chạy trên Server A.


In [ ]:
# ============================================================
# BƯỚC 22.3 — SERVER A: deploy 3 MON
# ============================================================

# Yêu cầu Ceph đặt 3 MON
sudo ceph orch apply mon 3

# Theo dõi service
sudo ceph orch ps --daemon-type mon

# Đợi 2–3 phút rồi kiểm tra
sudo ceph mon stat

# Expected:
# 3 mons at {server-a, server-b, server-c}


## BƯỚC 22.4 — Deploy MGR

MGR dùng cho dashboard, orchestrator, prometheus module, telemetry...

Thông thường lab 3 node nên chạy 2 MGR để có standby.


In [ ]:
# ============================================================
# BƯỚC 22.4 — SERVER A: deploy 2 MGR
# ============================================================

sudo ceph orch apply mgr 2

# Theo dõi MGR
sudo ceph orch ps --daemon-type mgr

# Kiểm tra MGR status
sudo ceph mgr stat

# Kiểm tra dashboard/prometheus service nếu có
sudo ceph mgr services


## BƯỚC 22.5 — Verify cluster sau khi thêm host

Ở thời điểm này có thể vẫn `HEALTH_WARN` vì chưa có OSD.

Điều quan trọng là:

- 3 host đã có trong orch host list.
- MON đã đủ 3.
- MGR có active/standby.


In [ ]:
# ============================================================
# VERIFY sau khi add host/MON/MGR
# ============================================================

sudo ceph -s
sudo ceph orch host ls
sudo ceph mon stat
sudo ceph mgr stat
sudo ceph orch ps

# Nếu host không add được:
# - kiểm tra SSH root
# - kiểm tra hostname
# - kiểm tra /etc/hosts
# - kiểm tra network 10.0.0.x hoặc 10.0.3.x


## BƯỚC 23 — Thêm OSD và tạo Pool cho OpenStack

## Mục tiêu

Sau khi có MON/MGR/host, thêm OSD từ `/dev/sda5` trên cả 3 server.

Sau đó tạo các pool phục vụ OpenStack:

| Pool | Dùng cho |
|---|---|
| `images` | Glance image |
| `volumes` | Cinder volumes |
| `vms` | Nova ephemeral / VM disk |
| `backups` | Cinder backup |

## Lưu ý về OSD disk

Trong lab này dùng:

```bash
/dev/sda5
```

trên cả 3 server.

Trước khi add OSD, kiểm tra:

```bash
sudo ceph orch device ls
```

Nếu `/dev/sda5` không hiện available, cần kiểm tra xem còn filesystem/LVM signature không.


## BƯỚC 23.1 — Xem device available

Chạy trên Server A.


In [ ]:
# ============================================================
# BƯỚC 23.1 — SERVER A: xem disk available cho OSD
# ============================================================

# Liệt kê device trên các host
sudo ceph orch device ls

# Xem chi tiết device rejected reason nếu có
sudo ceph orch device ls --wide

# Nếu /dev/sda5 bị reject:
# - Có filesystem signature
# - Có LVM signature
# - Đang mounted
# - Không đủ size
# - Ceph detect là locked/unavailable


## BƯỚC 23.2 — Add OSD từ `/dev/sda5`

Chạy trên Server A.

Lệnh này yêu cầu `/dev/sda5` trên từng host đã sạch.


In [ ]:
# ============================================================
# BƯỚC 23.2 — SERVER A: add OSD trên cả 3 server
# ============================================================

# Add OSD cho Server A
sudo ceph orch daemon add osd server-a:/dev/sda5

# Add OSD cho Server B
sudo ceph orch daemon add osd server-b:/dev/sda5

# Add OSD cho Server C
sudo ceph orch daemon add osd server-c:/dev/sda5

# Theo dõi OSD daemon
sudo ceph orch ps --daemon-type osd

# Kiểm tra OSD tree
sudo ceph osd tree

# Kiểm tra trạng thái cluster
sudo ceph -s


## BƯỚC 23.3 — Verify OSD

Mục tiêu:

- Có 3 OSD.
- Cả 3 OSD đều `up`.
- Cả 3 OSD đều `in`.
- Cluster có thể chuyển từ `HEALTH_WARN` sang `HEALTH_OK` sau khi PG ổn định.


In [ ]:
# ============================================================
# BƯỚC 23.3 — Verify OSD
# ============================================================

# Xem cây OSD
sudo ceph osd tree

# Xem dung lượng OSD
sudo ceph osd df

# Xem trạng thái OSD
sudo ceph osd stat

# Theo dõi cluster realtime
sudo ceph -w

# Ctrl+C để thoát ceph -w


## BƯỚC 23.4 — Tạo pool cho OpenStack

Chạy trên Server A.

Pool cần tạo:

- `images`
- `volumes`
- `vms`
- `backups`

PG number trong tài liệu:

- images: 32
- volumes: 32
- vms: 32
- backups: 16

Với lab 3 OSD, số PG này chấp nhận được cho mục đích học/lab.


In [ ]:
# ============================================================
# BƯỚC 23.4 — SERVER A: tạo Ceph pools cho OpenStack
# ============================================================

# Tạo pool images cho Glance
sudo ceph osd pool create images 32

# Tạo pool volumes cho Cinder
sudo ceph osd pool create volumes 32

# Tạo pool vms cho Nova
sudo ceph osd pool create vms 32

# Tạo pool backups cho Cinder Backup
sudo ceph osd pool create backups 16

# Verify danh sách pool
sudo ceph osd pool ls


## BƯỚC 23.5 — Enable application tag `rbd`

Ceph yêu cầu pool có application tag.

Vì OpenStack dùng RBD, ta enable application `rbd` cho cả 4 pool.


In [ ]:
# ============================================================
# BƯỚC 23.5 — Enable application rbd cho các pool
# ============================================================

sudo ceph osd pool application enable images rbd
sudo ceph osd pool application enable volumes rbd
sudo ceph osd pool application enable vms rbd
sudo ceph osd pool application enable backups rbd

# Verify application tag
sudo ceph osd pool application get images
sudo ceph osd pool application get volumes
sudo ceph osd pool application get vms
sudo ceph osd pool application get backups


## BƯỚC 23.6 — Set replica size

Với 3 OSD trên 3 server, replica size nên là `3`.

Điều này nghĩa là mỗi object có 3 bản sao.

Nếu lab thiếu OSD hoặc đang bootstrap chưa đủ, có thể tạm min_size thấp hơn, nhưng production nên giữ replica phù hợp.


In [ ]:
# ============================================================
# BƯỚC 23.6 — Set replica size cho pool
# ============================================================
# đoạn này ở trong cephadm shell nó là root@stack1 nên không cần dùng sudo
# Nếu chạy ngoài cephadm shell thì cần sudo, nhưng tốt nhất nên vào cephadm shell để thao tác với ceph CLI.
# bản chất stack@stack1 là nơi chạy: docker / podman, cephadm có thể KHÔNG có keyring, nên cần dùng sudo để đảm bảo có quyền truy cập keyring khi chạy ceph CLI.
# Nếu chạy ngoài cephadm shell, có thể cần export CEPH_KEYRING=/etc/ceph/ceph.client.admin.keyring để đảm bảo ceph CLI có thể tìm thấy keyring admin.

# ở đây xóa toàn bộ sudo chạy trong cephadm shell

# Set replica size = 3
ceph osd pool set images size 3
ceph osd pool set volumes size 3
ceph osd pool set vms size 3
ceph osd pool set backups size 3

# Set min_size = 2 cho an toàn hơn trong cụm 3 replica
ceph osd pool set images min_size 2
ceph osd pool set volumes min_size 2
ceph osd pool set vms min_size 2
ceph osd pool set backups min_size 2

# Verify pool detail
ceph osd pool get images size
ceph osd pool get volumes size
ceph osd pool get vms size
ceph osd pool get backups size

ceph osd pool get images min_size
ceph osd pool get volumes min_size
ceph osd pool get vms min_size
ceph osd pool get backups min_size


## BƯỚC 23.7 — Tạo Ceph user/keyring cho OpenStack

OpenStack service cần user Ceph riêng:

| Ceph client | Dùng cho | Pool quyền |
|---|---|---|
| `client.glance` | Glance image | `images` |
| `client.cinder` | Cinder volume | `volumes`, `vms`, `images` |
| `client.nova` | Nova compute | `vms`, `images` |

Keyring sẽ được lưu tại:

```bash
/etc/ceph/ceph.client.glance.keyring
/etc/ceph/ceph.client.cinder.keyring
/etc/ceph/ceph.client.nova.keyring
```


In [ ]:
# ============================================================
# BƯỚC 23.7 — Tạo Ceph users/keyrings cho OpenStack
# ============================================================


# User cho Glance
sudo ceph auth get-or-create client.glance \
  mon "profile rbd" \
  osd "profile rbd pool=images" \
  > /etc/ceph/ceph.client.glance.keyring

# User cho Cinder
sudo ceph auth get-or-create client.cinder \
  mon "profile rbd" \
  osd "profile rbd pool=volumes, profile rbd pool=vms, profile rbd pool=images" \
  > /etc/ceph/ceph.client.cinder.keyring

# User cho Nova
sudo ceph auth get-or-create client.nova \
  mon "profile rbd" \
  osd "profile rbd pool=vms, profile rbd pool=images" \
  > /etc/ceph/ceph.client.nova.keyring

# Set permission file
sudo chmod 600 /etc/ceph/ceph.client.*.keyring

# Verify auth
sudo ceph auth get client.glance
sudo ceph auth get client.cinder
sudo ceph auth get client.nova

# Verify files
sudo ls -lh /etc/ceph/ceph.client.*.keyring


## BƯỚC 23.8 — Copy Ceph config/keyring sang Controller VM

Cần copy sang 3 Controller VM:

- `controller-1`: `10.0.0.11`
- `controller-2`: `10.0.0.12`
- `controller-3`: `10.0.0.13`

Các file cần copy:

- `/etc/ceph/ceph.conf`
- `/etc/ceph/ceph.client.glance.keyring`
- `/etc/ceph/ceph.client.cinder.keyring`
- `/etc/ceph/ceph.client.nova.keyring`

Ban đầu copy vào `/tmp/`, sau này phần Kolla-Ansible sẽ đưa vào đúng `/etc/kolla/config/...`.


In [ ]:
# ============================================================
# BƯỚC 23.8 — Copy Ceph config/keyrings sang Controller VM
# Chạy trên Server A
# ============================================================

# Copy sang Controller 1
scp /etc/ceph/ceph.conf ubuntu@10.0.0.11:/tmp/
scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.11:/tmp/

# Copy sang Controller 2
scp /etc/ceph/ceph.conf ubuntu@10.0.0.12:/tmp/
scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.12:/tmp/

# Copy sang Controller 3
scp /etc/ceph/ceph.conf ubuntu@10.0.0.13:/tmp/
scp /etc/ceph/ceph.client.*.keyring ubuntu@10.0.0.13:/tmp/


## BƯỚC 23.9 — Verify trên Controller VM

SSH vào từng Controller VM và kiểm tra file đã copy sang `/tmp`.

Chạy trong từng Controller VM.


In [ ]:
# ============================================================
# Chạy trong từng Controller VM
# ============================================================

#ssh sang Controller VM trước rồi chạy các lệnh dưới đây
ssh ubuntu@10.0.0.13

# Kiểm tra file Ceph đã được copy
ls -lh /tmp/ceph.conf
ls -lh /tmp/ceph.client.*.keyring

# Xem nội dung ceph.conf
cat /tmp/ceph.conf

# Nếu đã cài ceph-common trong Controller VM, test thử:
which ceph || true
ceph -c /tmp/ceph.conf -s || true

# Nếu ceph -s chưa chạy được ở đây, chưa chắc là lỗi.
# Có thể cần copy keyring/admin hoặc cấu hình client đúng sau này trong Kolla.


## BƯỚC 24 — Verify tổng thể Ceph cluster

Sau khi thêm OSD và tạo pool, chạy các lệnh dưới đây trên Server A.


In [ ]:
# ============================================================
# BƯỚC 24 — VERIFY tổng thể Ceph
# ============================================================

# Health tổng thể
sudo ceph -s

# Chi tiết health nếu có WARN
sudo ceph health detail

# Host list
sudo ceph orch host ls

# Service list
sudo ceph orch ps

# MON/MGR
sudo ceph mon stat
sudo ceph mgr stat

# OSD
sudo ceph osd tree
sudo ceph osd df
sudo ceph osd stat

# Pool
sudo ceph osd pool ls
sudo ceph df

# Dashboard URL
sudo ceph mgr services


#Nếu không vào được dashboard thì có thể do firewall hoặc do dashboard bind IP mặc định là localhost. Có thể chạy lệnh dưới để bind dashboard ra
sudo ceph config set mgr mgr/dashboard/server_addr 0.0.0.0
sudo ceph mgr module disable dashboard
sudo ceph mgr module enable dashboard
#Nếu quên mật khẩu admin thì có thể reset lại bằng lệnh dưới
sudo ceph dashboard ac-user-show
echo "123456" | sudo ceph dashboard ac-user-set-password admin -i -

ceph
tk: admin
mk:ubuntu123

## Debug nhanh lỗi thường gặp

### 1. `cephadm bootstrap` fail vì network

Kiểm tra:

```bash
ip addr
ip route
ping -c 3 10.0.0.22
ping -c 3 10.0.0.23
ping -c 3 10.0.3.22
ping -c 3 10.0.3.23
```

Nếu `10.0.3.x` chưa ổn định, dùng bootstrap bằng `10.0.0.x`.

### 2. Add host fail vì SSH root

Kiểm tra:

```bash
sudo ceph cephadm get-pub-key
sudo ssh -i /etc/ceph/cephadm-key root@10.0.0.22 hostname
```

Nếu root SSH bị disable, cần bật tạm hoặc copy key thủ công vào:

```bash
/root/.ssh/authorized_keys
```

trên Server B/C.

### 3. OSD không add được

Kiểm tra:

```bash
sudo ceph orch device ls --wide
sudo wipefs -n /dev/sda5
sudo pvs | grep sda5
findmnt /dev/sda5
```

Nếu còn signature:

```bash
sudo wipefs -a /dev/sda5
sudo sgdisk --zap-all /dev/sda5 || true
sudo partprobe /dev/sda
```

### 4. Ceph báo clock skew

Trên cả 3 server:

```bash
chronyc tracking
chronyc sources -v
timedatectl
```

Restart chrony nếu cần:

```bash
sudo systemctl restart chrony
```

### 5. Dashboard không vào được

Kiểm tra URL:

```bash
sudo ceph mgr services
```

Kiểm tra port:

```bash
sudo ss -lntp | grep 8443
```

Nếu dùng network nội bộ, máy ngoài có thể không truy cập được dashboard nếu không route tới `10.0.0.x` hoặc `10.0.3.x`.

### 6. Pool chưa active+clean

Theo dõi:

```bash
sudo ceph -w
sudo ceph pg stat
sudo ceph health detail
```

Chờ Ceph peering/recovery xong.


## Checklist hoàn thành PHẦN V

Trước khi chuyển sang tích hợp OpenStack/Kolla-Ansible với Ceph, kiểm tra đủ:

- [ ] Cả 3 server đã cài `ceph-common`, `lvm2`, `gdisk`, `smartmontools`, `chrony`.
- [ ] `/etc/hosts` có `server-a`, `server-b`, `server-c`.
- [ ] Server A SSH được sang Server B/C.
- [ ] Chrony hoạt động, không clock skew lớn.
- [ ] Xác định đúng OSD partition là `/dev/sda5`.
- [ ] Không wipe nhầm OS partition hoặc VM thin-pool.
- [ ] Cephadm đã cài trên Server A.
- [ ] Bootstrap Ceph thành công.
- [ ] Có 3 host trong `ceph orch host ls`.
- [ ] Có 3 MON.
- [ ] Có 2 MGR.
- [ ] Có 3 OSD up/in.
- [ ] `ceph -s` đạt `HEALTH_OK` hoặc chỉ còn WARN đã hiểu nguyên nhân.
- [ ] Pool `images`, `volumes`, `vms`, `backups` đã tạo.
- [ ] Pool đã enable application `rbd`.
- [ ] Replica size/min_size đã set.
- [ ] Keyring `client.glance`, `client.cinder`, `client.nova` đã tạo.
- [ ] Ceph config/keyring đã copy sang Controller VM1/2/3.
